# 02 - Nettoyage et qualité des données

On applique `clean_games` (dans `src/cleaning.py`) et on documente ce que chaque filtre retire.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.data_loader import load_games
from src.cleaning import RAW_TO_CLEAN, clean_games, parse_date, parse_owners, clean_tag_string, load_clean

## 1. Tester les petites fonctions à la main

In [2]:
print(parse_date('Aug 1, 2023'))
print(parse_date('pas une date'))       # None, sans planter
print(parse_owners('20000 - 50000'))
print(parse_owners(None))               # (None, None), sans planter
print(clean_tag_string(' Indie , Roguelike,, Action '))

2023-08-01 00:00:00
None
(20000, 50000)
(None, None)
Indie,Roguelike,Action


## 2. Charger et nettoyer

In [3]:
raw = load_games('../data/games.csv', usecols=list(RAW_TO_CLEAN.keys()))
df, report = clean_games(raw)
df.shape

(71352, 10)

## 3. Rapport de qualité

In [4]:
report

,étape,lignes retirées,% du brut retiré,lignes restantes,raison
0,Départ (fichier brut),0,0.0,125855,
1,Colonnes du contrat,0,0.0,125855,"Garder les colonnes utiles, renommées en snake..."
2,Doublons et sans nom,1,0.0,125854,Un jeu compté deux fois fausse les statistiques
3,Remise de 100 %,6,0.0,125848,Prix plein impossible à retrouver (le prix sol...
4,Jeux gratuits,26656,21.2,99192,"Sans prix, pas de revenu de vente à estimer"
5,Dates illisibles,0,0.0,99192,Impossible de situer le jeu dans le temps
6,Owners illisibles,0,0.0,99192,Impossible de valider l'estimation des ventes
7,Jeux sans tag,24478,19.4,74714,Impossible de les comparer à un concept
8,Sortis il y a moins de 6 mois,247,0.2,74467,Leurs avis n'ont pas eu le temps d'arriver
9,Sortis avant 2015,3115,2.5,71352,Le marché d'avant 2015 ne ressemble pas à celu...


In [5]:
# Vérification : lignes de départ - lignes retirées = lignes finales
start = report['lignes restantes'].iloc[0]
removed = report['lignes retirées'].sum()
print(start, '-', removed, '=', start - removed, '| lignes finales :', len(df))
assert start - removed == len(df)

125855 - 54503 = 71352 | lignes finales : 71352


Les deux plus gros filtres sont les jeux gratuits (26 656) et les jeux sans tag (24 478).
Au final, on garde 71 352 jeux payants, tagués, sortis entre 2015 et août 2025.

**Piège n°2** : la colonne `Price` est le prix du jour de la collecte, **remise comprise** (41 % des jeux payants étaient en promotion).
`restore_full_price` retrouve le prix plein : prix / (1 - remise / 100). Les 6 jeux à -100 % sont retirés (prix plein impossible à retrouver).

In [6]:
raw_hk = raw[raw['Name'] == 'Hollow Knight'][['Name', 'Price', 'Discount']]
clean_hk = df[df['name'] == 'Hollow Knight'][['name', 'price']]
print(raw_hk.to_string(index=False))
print(clean_hk.to_string(index=False))

         Name  Price  Discount
Hollow Knight   7.49        50
         name  price
Hollow Knight  14.98


Hollow Knight : 7,49 $ soldé à -50 % dans le fichier brut, 14,98 $ de prix plein après correction (14,99 $ en réalité : l'écart vient de l'arrondi du prix soldé).
**Choix de l'année 2015** : avant, Steam était un catalogue filtré (moins de 2 500 sorties par an) ; le marché ne ressemble pas à celui d'aujourd'hui.

## 4. strptime contre pd.to_datetime

In [7]:
dates_strptime = pd.to_datetime(raw['Release date'].apply(parse_date))
dates_pandas = pd.to_datetime(raw['Release date'], format='%b %d, %Y', errors='coerce')
print('Résultats identiques :', (dates_strptime == dates_pandas).all())

Résultats identiques : True


Les deux méthodes donnent la même chose. `pd.to_datetime` est plus rapide ; la version `strptime` + `try/except` montre explicitement ce qu'on fait des dates illisibles.

## 5. Sauvegarder, recharger, vérifier

In [8]:
df.to_csv('../data/games_clean.csv', index=False)
clean = load_clean('../data/games_clean.csv')
clean.dtypes

app_id                   int64
name                       str
release_date    datetime64[us]
release_year             int64
price                  float64
positive                 int64
negative                 int64
owners_min               int64
owners_max               int64
tags                       str
dtype: object

In [9]:
print('Prix à 0 :', (clean['price'] == 0).sum())
print('Années :', clean['release_year'].min(), 'à', clean['release_year'].max())
print('owners_min <= owners_max partout :', (clean['owners_min'] <= clean['owners_max']).all())
clean.head()

Prix à 0 : 0
Années : 2015 à 2025
owners_min <= owners_max partout : True


,app_id,name,release_date,release_year,price,positive,negative,owners_min,owners_max,tags
0,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,2016,14.97,252,3,0,20000,"Adventure,Visual Novel,Anime,Cute"
1,1034400,Mystery Solitaire The Black Raven,2019-05-06,2019,4.99,21,3,0,20000,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec..."
2,1934300,Armored Brigade II,2025-04-08,2025,39.99,117,13,0,20000,"Simulation,Strategy,RTS,Wargame,Grand Strategy..."
3,1157670,Hepta Beats,2021-05-07,2021,0.99,28,4,0,20000,"Precision Platformer,Rhythm,Music,Difficult,Ab..."
4,1540330,MUMBA IV: Egypt Jewels,2021-12-13,2021,0.59,19,6,0,20000,"Casual,Action,Strategy,Puzzle,Match 3,Singlepl..."
